# 04 · Model Inference
How the client's team would use the saved model: score **new sessions** (single session or a batch export) and get a probability, a yes/no prediction and a recommended marketing action.

In [1]:
import sys
sys.path.append("..")
import pandas as pd
from src.predict import load_model, predict_df

model, meta = load_model()
print("Model:", meta["model"], "| threshold:", meta["threshold"])
print("Test metrics:", meta["test_metrics"])

Model: HistGradientBoostingClassifier (tuned) | threshold: 0.684
Test metrics: {'f1': 0.6695, 'precision': 0.6346, 'recall': 0.7086, 'accuracy': 0.8791, 'pr_auc': 0.7504, 'roc_auc': 0.9064}


## 1. A single live session
A returning visitor from São Paulo, on mobile, coming from a WhatsApp campaign during Dia das Mães, who added one item to the cart and checked the installment options.

In [2]:
session = pd.DataFrame([{
    "pages_viewed": 14, "category_pages": 3, "product_pages": 8, "searches": 1,
    "session_duration_sec": 520, "product_page_duration_sec": 260, "add_to_cart_count": 1,
    "wishlist_adds": 0, "cart_value_brl": 289.90, "bounce_rate": 0.08, "exit_rate": 0.21,
    "page_value_brl": 55.0, "hour": 21, "month": 5,
    "viewed_freight_calculator": 0, "viewed_installment_options": 1, "coupon_applied": 0, "is_weekend": 0,
    "device_type": "Mobile", "traffic_source": "WhatsApp", "visitor_type": "Returning",
    "region": "Sudeste", "sales_event": "Dia das Mães",
}])
predict_df(model, meta, session)[["purchase_probability", "predicted_purchase", "intent_segment", "recommended_action"]]

,purchase_probability,predicted_purchase,intent_segment,recommended_action
0,0.9242,1,High intent,Likely to buy — don't give away a discount; show free-shipping threshold / installment banner.


## 2. Batch scoring of a new export

In [3]:
new = pd.read_csv("../data/sample/new_sessions.csv")
scored = predict_df(model, meta, new)
scored[["session_id", "traffic_source", "device_type", "add_to_cart_count", "page_value_brl",
        "purchase_probability", "intent_segment"]].sort_values("purchase_probability", ascending=False).head(10)

,session_id,traffic_source,device_type,add_to_cart_count,page_value_brl,purchase_probability,intent_segment
18,CE-3342204,Google Orgânico,Mobile,2,104.86,0.9681,High intent
5,CE-2708750,Google Orgânico,Desktop,0,74.16,0.8335,High intent
19,CE-3890741,WhatsApp,Desktop,3,0.00,0.7855,High intent
9,CE-4781486,WhatsApp,Desktop,3,20.82,0.6925,High intent
12,CE-6179237,Google Ads,Mobile,3,0.00,0.6150,Medium intent
11,CE-9316674,Google Orgânico,Mobile,1,0.00,0.5159,Medium intent
0,CE-3421555,Instagram,Mobile,0,97.56,0.4738,Medium intent
14,CE-5885473,Direto,Mobile,3,0.00,0.4309,Medium intent
2,CE-9156816,Google Orgânico,Mobile,0,0.00,0.2427,Low intent
7,CE-8982311,WhatsApp,Mobile,1,0.00,0.1710,Low intent


In [4]:
scored["intent_segment"].value_counts()

,count
intent_segment,
Low intent,17
Medium intent,4
High intent,4


In [5]:
scored.to_csv("../data/sample/new_sessions_scored.csv", index=False)
print("Saved → data/sample/new_sessions_scored.csv")

Saved → data/sample/new_sessions_scored.csv
